In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Step 1: Generate synthetic data for classification
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_classes=2,
    random_state=42,
    flip_y=0.15
)

In [ ]:
# =========================================================
# Fixed-C Cross-Validation
# =========================================================

# Step 2: Fixed CV setup
c_value = 1
n_folds = 5
cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
accuracies = []

In [ ]:

# Step 3: Perform fixed CV
fold = 0
for train_index, test_index in cv.split(X, y):
    fold += 1

    # Split
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Training
    model = LogisticRegression(C=c_value, random_state=42, max_iter=1000, solver='liblinear')
    model.fit(X_train, y_train)

    # Testing
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracies.append(accuracy)

    print(f"Fixed CV - Fold {fold}: Accuracy = {accuracy:.4f}")

print(f"\nAverage Fixed-C Accuracy: {np.mean(accuracies):.4f}")

# Step 4: Plot fixed CV accuracies
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_folds + 1), accuracies, marker='o')
plt.title('Fixed-C Accuracy vs Fold')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

In [ ]:
# =========================================================
# Nested Cross-Validation
# =========================================================

# Step 5: Nested CV setup
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

outer_accuracies = []
best_cs = []

In [ ]:
# Step 6: Perform nested CV
for outer_fold, (train_index, test_index) in enumerate(outer_cv.split(X, y), start=1):
    # Outer split
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    best_c = None
    best_score = -np.inf

    # Loop over candidate C values
    for c in param_grid['C']:
        inner_accuracies = []

        # Inner CV for current C
        for inner_train_index, inner_val_index in inner_cv.split(X_train, y_train):
            # Inner split
            X_train_fold = X_train[inner_train_index]
            X_val_fold = X_train[inner_val_index]
            y_train_fold = y_train[inner_train_index]
            y_val_fold = y_train[inner_val_index]

            # Inner training
            model = LogisticRegression(C=c, random_state=42, max_iter=1000, solver='liblinear')
            model.fit(X_train_fold, y_train_fold)

            # Inner validation
            y_val_pred = model.predict(X_val_fold)
            acc = accuracy_score(y_val_fold, y_val_pred)
            inner_accuracies.append(acc)

        # Mean inner accuracy for this C
        mean_acc = np.mean(inner_accuracies)

        # Update best C if this one performs better
        if mean_acc > best_score:
            best_score = mean_acc
            best_c = c

    best_cs.append(best_c)

    # Train outer model with best C
    final_model = LogisticRegression(C=best_c, random_state=42, max_iter=1000, solver='liblinear')
    final_model.fit(X_train, y_train)
    y_test_pred = final_model.predict(X_test)

    outer_acc = accuracy_score(y_test, y_test_pred)
    outer_accuracies.append(outer_acc)

    print(f"Nested CV - Outer Fold {outer_fold}: Accuracy = {outer_acc:.4f} | Best C = {best_c}")

# Final results
print("\nFinal Nested CV Results:")
print(f"Average Outer Accuracy: {np.mean(outer_accuracies):.4f}")
print(f"Best C values per outer fold: {best_cs}")